# SpeakerForge Phase 2 — CSM-1B LoRA 训练 (Colab T4)

**前提：**
1. 本地已运行 `stage1`，生成 `sesame/data/Akinokoe_vB/`
   - stage1 从 `../speakerforge/dataset/Akinokoe_versionB/` 读取数据（Phase 1 stage7 输出）
   - stage1 将音频以 WAV bytes 嵌入 Parquet，数据集完全自包含，可移植到 Colab
2. 已将 `sesame/data/Akinokoe_vB/` 复制到 Google Drive（如 `My Drive/SpeakerForge/sesame/data/Akinokoe_vB/`）
3. HuggingFace read token 已添加到 Colab Secrets（key 名：`HF_TOKEN`）

**数据集格式（以 notebook preprocess_example 为依据）：**
- `audio`：HF Audio feature，24kHz，float32
- `text`：str 转录文本
- `source`：str 说话人ID（单说话人默认 "0"，在 Cell 4 自动添加）

训练完成后，adapter 保存到 Drive，本地直接通过 G: 盘访问，然后运行 `stage3`。

In [1]:
# ── Cell 1: 安装依赖 ──────────────────────────────────────────────────────────

import os, re
import torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.52.3
!pip install --no-deps trl==0.22.2
!pip install torchcodec "datasets>=3.4.1,<4.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 56.0 MB/s eta 0:00:00
   ━

In [2]:
# ── Cell 2: 挂载 Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# 修改为你 Drive 中实际的路径
DRIVE_ROOT = "/content/drive/MyDrive/SpeakerForge/sesame"
DATA_DIR   = f"{DRIVE_ROOT}/data/Akinokoe_vB"      # stage1 输出
OUTPUT_DIR = f"{DRIVE_ROOT}/models/Akinokoe"        # adapter 保存位置

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data:   {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Exists: {os.path.exists(DATA_DIR)}")

Mounted at /content/drive
Data:   /content/drive/MyDrive/SpeakerForge/sesame/data/Akinokoe_vB
Output: /content/drive/MyDrive/SpeakerForge/sesame/models/Akinokoe
Exists: True


In [3]:
# ── Cell 3: 加载模型 + LoRA ───────────────────────────────────────────────────
from unsloth import FastModel
from transformers import CsmForConditionalGeneration
import inspect
import transformers.models.csm.modeling_csm as _csm_mod

MODEL_NAME = "unsloth/csm-1b"

model, processor = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    auto_model=CsmForConditionalGeneration,
    load_in_4bit=False,
)

# ── Patch BEFORE get_peft_model ───────────────────────────────────────────────
# Unsloth GC 在 get_peft_model 时把 forward 存入 closure；必须先 patch。
# 注意：不能用 functools.wraps，否则 inspect.unwrap() 会绕过 wrapper 直接调原函数。
_target_cls = None
for _name in dir(_csm_mod):
    _obj = getattr(_csm_mod, _name)
    if isinstance(_obj, type) and hasattr(_obj, 'forward'):
        _params = list(inspect.signature(_obj.forward).parameters)
        if 'backbone_last_hidden_state' in _params and 'inputs_embeds' in _params:
            _target_cls = _obj
            print(f"Patching: {_name}")
            break

if _target_cls is None:
    print("WARNING: depth decoder class not found — patch not applied")
else:
    _orig_depth_fwd = _target_cls.forward
    def _patched_depth_fwd(self, *args, **kwargs):
        if 'inputs_embeds' in kwargs:
            if kwargs['inputs_embeds'] is not None:
                kwargs['inputs_embeds'] = kwargs['inputs_embeds'].clone()
        elif len(args) > 5 and args[5] is not None:
            args = args[:5] + (args[5].clone(),) + args[6:]
        return _orig_depth_fwd(self, *args, **kwargs)
    _target_cls.forward = _patched_depth_fwd
    print("Patch applied.")

# ── LoRA — 与官方对齐 ─────────────────────────────────────────────────────────
model = FastModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print("Model loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py:404: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Csm patching. Transformers: 4.52.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/4.15G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/csm-1b does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Patching: CsmDepthDecoderForCausalLM
Patch applied.
Model loaded.


In [4]:
# ── Cell 4: 数据集预处理 ──────────────────────────────────────────────────────
# 与官方对齐：显式用 AutoProcessor（官方 Cell 10 会覆盖 FastModel 返回的 processor）
import torch
from datasets import load_from_disk, Audio
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_NAME)

raw_ds = load_from_disk(DATA_DIR)
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=24000))
if "source" not in raw_ds.column_names:
    raw_ds = raw_ds.add_column("source", ["0"] * len(raw_ds))
print(f"Raw dataset: {len(raw_ds)} samples")

def preprocess_example(example):
    conversation = [{
        "role": example["source"],
        "content": [
            {"type": "text",  "text":  example["text"]},
            {"type": "audio", "path":  example["audio"]["array"]},
        ],
    }]
    try:
        model_inputs = processor.apply_chat_template(
            conversation, tokenize=True, return_dict=True, output_labels=True,
            text_kwargs={"padding": "max_length", "max_length": 256,
                         "pad_to_multiple_of": 8, "padding_side": "right"},
            audio_kwargs={"sampling_rate": 24_000, "max_length": 240001,
                          "padding": "max_length"},
            common_kwargs={"return_tensors": "pt"},
        )
    except Exception as e:
        print(f"  Skip '{example['text'][:40]}': {e}")
        return None
    required = ["input_ids", "attention_mask", "labels", "input_values", "input_values_cutoffs"]
    result = {k: model_inputs[k][0] for k in required if k in model_inputs}
    if len(result) < len(required):
        return None
    return result

processed_ds = raw_ds.map(
    preprocess_example,
    remove_columns=raw_ds.column_names,
    desc="Preprocessing",
)
processed_ds = processed_ds.filter(lambda ex: ex is not None)
print(f"Processed: {len(processed_ds)} samples")

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Raw dataset: 1492 samples
Processed: 1492 samples


In [5]:
# ── Cell 5: 训练 ──────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported

MAX_STEPS = 120

trainer = Trainer(
    model=model,
    train_dataset=processed_ds,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        remove_unused_columns=False,  # CSM audio columns not in standard forward sig
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name}, {gpu_stats.total_memory/1024**3:.1f} GB")

trainer_stats = trainer.train()
print(f"\nTraining done: {trainer_stats.metrics['train_runtime']:.0f}s  "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,492 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,032,448 of 1,661,132,609 (1.75% trained)


GPU: Tesla T4, 14.6 GB
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,9.173500
2,8.996600
3,8.942000
4,8.928400
5,8.794400
6,8.638200
7,8.554100
8,8.652200
9,8.588400
10,8.472300



Training done: 585s  (9.7 min)


In [ ]:
# ── Cell 6: 保存 adapter 到 Drive ────────────────────────────────────────────
adapter_path = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)
print(f"Adapter saved → {adapter_path}")
print("\n本地 G: 盘路径（可直接用于 stage3）：")
print(adapter_path.replace("/content/drive/MyDrive", "G:").replace("/", "\\"))

In [80]:
# ── Cell 7 (v4): 生成测试 ─────────────────────────────────────────────────────
import ast, textwrap, inspect, types
import numpy as np
import transformers.generation.utils as _gen_utils
import transformers.models.csm.modeling_csm as _csm_mod

# ── Patch 1: skip CSM-specific kwargs in _validate_model_kwargs ───────────────
_CSM_SKIP_VALIDATE = {'backbone_last_hidden_state', 'input_values', 'input_values_cutoffs'}

if not hasattr(_gen_utils.GenerationMixin, '_csm_orig_validate_model_kwargs'):
    _gen_utils.GenerationMixin._csm_orig_validate_model_kwargs = (
        _gen_utils.GenerationMixin._validate_model_kwargs
    )
    def _global_validate(self, model_kwargs):
        for k in _CSM_SKIP_VALIDATE:
            model_kwargs.pop(k, None)
        return _gen_utils.GenerationMixin._csm_orig_validate_model_kwargs(self, model_kwargs)
    _gen_utils.GenerationMixin._validate_model_kwargs = _global_validate
    print("Patch 1 applied")
else:
    print("Patch 1 already active")

# ── Patch 2: fix prepare_inputs_for_generation (instance-level) ───────────────
# Fix A: model_inputs.pop("position_ids") -> KeyError in transformers 4.52.3
# Fix B: super() without args needs __class__ cell, absent when exec()'d
# Fix C: bind at instance level so Unsloth class re-patches during generate() lose
_DepthDecoder = _csm_mod.CsmDepthDecoderForCausalLM
_filepath = inspect.getfile(_csm_mod)
with open(_filepath, 'r') as _f:
    _file_src = _f.read()

_tree = ast.parse(_file_src)
_func_node = None
for _node in ast.walk(_tree):
    if isinstance(_node, ast.ClassDef) and _node.name == 'CsmDepthDecoderForCausalLM':
        for _item in _node.body:
            if isinstance(_item, ast.FunctionDef) and _item.name == 'prepare_inputs_for_generation':
                _func_node = _item
                break
        break

_dd_instance = None
if _func_node is None:
    print("ERROR: prepare_inputs_for_generation not found in source")
else:
    _lines = _file_src.splitlines()
    _func_src = '\n'.join(_lines[_func_node.lineno - 1 : _func_node.end_lineno])
    _func_src = textwrap.dedent(_func_src)
    _func_src = _func_src.replace(
        'model_inputs.pop("position_ids")',
        'model_inputs.pop("position_ids", None)',
        1
    )
    _func_src = _func_src.replace(
        'super()',
        'super(CsmDepthDecoderForCausalLM, self)'
    )
    _ns = {**vars(_csm_mod), '__builtins__': __builtins__}
    exec(compile(_func_src, '<csm_depth_patch_v4>', 'exec'), _ns)
    _fixed_fn = _ns['prepare_inputs_for_generation']
    _DepthDecoder.prepare_inputs_for_generation = _fixed_fn

    for _name, _mod in model.named_modules():
        if isinstance(_mod, _DepthDecoder):
            _dd_instance = _mod
            break

    if _dd_instance is not None:
        _dd_instance.prepare_inputs_for_generation = types.MethodType(_fixed_fn, _dd_instance)
        print(f"Patch 2 applied (class + instance: {type(_dd_instance).__name__})")
    else:
        print("Patch 2 applied (class only -- instance not found)")

# ── Patch 3: fix Unsloth misc.py logits_to_keep=0 decode-step empty-tensor bug ─
# misc.py _full_forward always uses slice(1,None) when logits_to_keep==0 to skip
# the backbone hidden-state token at idx 0.  Prefill: seq_len>=2, [:,1:] keeps
# the real token. Decode: seq_len==1, [:,1:] -> empty [1,0,H] and
# codebook_indices[1:] -> empty [] -> CsmCodebooksHead_forward range(0) -> crash.
#
# Fix: wrap _dd.model to capture full hidden_states + cache_position BEFORE the
# bad slice; wrap _dd.codebooks_head to substitute them when empty arrives.
if _dd_instance is not None:
    _p3 = {'full_hs': None, 'cp': None}
    _dd_model = _dd_instance.model
    _dd_cbh   = _dd_instance.codebooks_head

    _p3_model_cls_fwd = type(_dd_model).forward
    def _p3_capture(self_m, *a, **kw):
        out = _p3_model_cls_fwd(self_m, *a, **kw)
        _p3['full_hs'] = out[0]
        _p3['cp'] = kw.get('cache_position', None)
        return out
    _dd_model.forward = types.MethodType(_p3_capture, _dd_model)

    _p3_cbh_cls_fwd = type(_dd_cbh).forward
    def _p3_fix_cbh(self_c, hidden_states, cache_position=None):
        if hidden_states.shape[1] == 0 and _p3['full_hs'] is not None:
            hidden_states = _p3['full_hs']
            cache_position = _p3['cp']
        return _p3_cbh_cls_fwd(self_c, hidden_states, cache_position=cache_position)
    _dd_cbh.forward = types.MethodType(_p3_fix_cbh, _dd_cbh)

    print("Patch 3 applied: misc.py decode empty-tensor fix")
else:
    print("Patch 3 skipped: _dd_instance not found")

# ── 推理 ──────────────────────────────────────────────────────────────────────
test_text = "Sesame is a super cool TTS model which can be fine tuned with Unsloth."
speaker_id = 0
conversation = [{"role": str(speaker_id), "content": [
    {"type": "text",  "text": test_text},
    {"type": "audio", "path": np.zeros(24000, dtype=np.float32)},
]}]

inputs = processor.apply_chat_template(
    conversation, tokenize=True, return_dict=True,
    common_kwargs={"return_tensors": "pt"},
    audio_kwargs={"sampling_rate": 24000},
).to("cuda")

audio = model.generate(**inputs, max_new_tokens=125)
print(f"generate() output shape: {audio.shape}")  # 预期 (1, frames, 32)

# generate() 返回 codec token IDs (batch, frames, codebooks)
# 需经 Mimi audio tower 解码为 waveform
_base = model.base_model.model if hasattr(model, 'base_model') else model

# 调试：找 audio_tower 位置
for name, mod in _base.named_children():
    print(name, type(mod).__name__)
_codes = audio.long().permute(0, 2, 1)  # (batch, codebooks, frames)
with torch.no_grad():
    _wav_out = _base.codec_model.half().decode(_codes)
_waveform = _wav_out.audio_values[0, 0].cpu().float().numpy()  # (samples,)

import soundfile as sf
sf.write("output.wav", _waveform, 24000)
print(f"Saved: output.wav  ({len(_waveform)/24000:.1f}s @ 24kHz)")
from IPython.display import Audio, display
display(Audio("output.wav", rate=24000))

Patch 1 already active
Patch 2 applied (class + instance: CsmDepthDecoderForCausalLM)
Patch 3 applied: misc.py decode empty-tensor fix
generate() output shape: torch.Size([1, 68, 32])
lm_head Linear
embed_text_tokens Embedding
backbone_model CsmBackboneModel
depth_decoder CsmDepthDecoderForCausalLM
codec_model MimiModel
Saved: output.wav  (5.4s @ 24kHz)


In [ ]:
带有上下文数据集

In [71]:
# ── Cell 7 (v4 final): 生成测试 ──────────────────────────────────────────────
import ast, textwrap, inspect, types
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
import transformers.generation.utils as _gen_utils
import transformers.models.csm.modeling_csm as _csm_mod

# ── Patch 1 ───────────────────────────────────────────────────────────────────
_CSM_SKIP_VALIDATE = {'backbone_last_hidden_state', 'input_values', 'input_values_cutoffs'}
if not hasattr(_gen_utils.GenerationMixin, '_csm_orig_validate_model_kwargs'):
    _gen_utils.GenerationMixin._csm_orig_validate_model_kwargs = (
        _gen_utils.GenerationMixin._validate_model_kwargs
    )
    def _global_validate(self, model_kwargs):
        for k in _CSM_SKIP_VALIDATE:
            model_kwargs.pop(k, None)
        return _gen_utils.GenerationMixin._csm_orig_validate_model_kwargs(self, model_kwargs)
    _gen_utils.GenerationMixin._validate_model_kwargs = _global_validate
    print("Patch 1 applied")
else:
    print("Patch 1 already active")

# ── Patch 2 ───────────────────────────────────────────────────────────────────
_DepthDecoder = _csm_mod.CsmDepthDecoderForCausalLM
_filepath = inspect.getfile(_csm_mod)
with open(_filepath, 'r') as _f:
    _file_src = _f.read()
_tree = ast.parse(_file_src)
_func_node = None
for _node in ast.walk(_tree):
    if isinstance(_node, ast.ClassDef) and _node.name == 'CsmDepthDecoderForCausalLM':
        for _item in _node.body:
            if isinstance(_item, ast.FunctionDef) and _item.name == 'prepare_inputs_for_generation':
                _func_node = _item
                break
        break
_dd_instance = None
if _func_node is None:
    print("ERROR: prepare_inputs_for_generation not found")
else:
    _lines = _file_src.splitlines()
    _func_src = '\n'.join(_lines[_func_node.lineno - 1 : _func_node.end_lineno])
    _func_src = textwrap.dedent(_func_src)
    _func_src = _func_src.replace('model_inputs.pop("position_ids")',
                                   'model_inputs.pop("position_ids", None)', 1)
    _func_src = _func_src.replace('super()', 'super(CsmDepthDecoderForCausalLM, self)')
    _ns = {**vars(_csm_mod), '__builtins__': __builtins__}
    exec(compile(_func_src, '<csm_depth_patch_v4>', 'exec'), _ns)
    _fixed_fn = _ns['prepare_inputs_for_generation']
    _DepthDecoder.prepare_inputs_for_generation = _fixed_fn
    for _name, _mod in model.named_modules():
        if isinstance(_mod, _DepthDecoder):
            _dd_instance = _mod
            break
    if _dd_instance is not None:
        _dd_instance.prepare_inputs_for_generation = types.MethodType(_fixed_fn, _dd_instance)
        print(f"Patch 2 applied (class + instance: {type(_dd_instance).__name__})")
    else:
        print("Patch 2 applied (class only)")

# ── Patch 3 ───────────────────────────────────────────────────────────────────
if _dd_instance is not None:
    _p3 = {'full_hs': None, 'cp': None}
    _dd_model = _dd_instance.model
    _dd_cbh   = _dd_instance.codebooks_head
    _p3_model_cls_fwd = type(_dd_model).forward
    def _p3_capture(self_m, *a, **kw):
        out = _p3_model_cls_fwd(self_m, *a, **kw)
        _p3['full_hs'] = out[0]
        _p3['cp'] = kw.get('cache_position', None)
        return out
    _dd_model.forward = types.MethodType(_p3_capture, _dd_model)
    _p3_cbh_cls_fwd = type(_dd_cbh).forward
    def _p3_fix_cbh(self_c, hidden_states, cache_position=None):
        if hidden_states.shape[1] == 0 and _p3['full_hs'] is not None:
            hidden_states = _p3['full_hs']
            cache_position = _p3['cp']
        return _p3_cbh_cls_fwd(self_c, hidden_states, cache_position=cache_position)
    _dd_cbh.forward = types.MethodType(_p3_fix_cbh, _dd_cbh)
    print("Patch 3 applied")
else:
    print("Patch 3 skipped")

# ── 推理：用数据集真实音频作 speaker context ──────────────────────────────────
speaker_id = 0

ref = raw_ds[3]
utterance      = ref["audio"]["array"]   # float32 numpy array @ 24kHz
utterance_text = ref["text"]
text = "Sesame is a super cool TTS model which can be fine tuned with Unsloth."

print(f"Reference: [{utterance_text[:60]}]")
print(f"Generate:  [{text}]")

conversation = [
    {"role": str(speaker_id), "content": [
        {"type": "text",  "text": utterance_text},
        {"type": "audio", "path": utterance},
    ]},
    {"role": str(speaker_id), "content": [
        {"type": "text",  "text": text},
    ]},
]

inputs = processor.apply_chat_template(
    conversation,
    tokenize=True,
    return_dict=True,
    common_kwargs={"return_tensors": "pt"},
    audio_kwargs={"sampling_rate": 24000},
).to("cuda")
#with model.disable_adapter():
audio_values = model.generate(
    **inputs,
    max_new_tokens=125,
    output_audio=True,       # 让模型内部做 codec decode，直接返回 waveform
)

audio = audio_values[0].to(torch.float32).cpu().numpy()
sf.write("output_with_context.wav", audio, 24000)
print(f"Saved: output_with_context.wav  ({len(audio)/24000:.1f}s)")
display(Audio(audio, rate=24000))

Patch 1 already active
Patch 2 applied (class + instance: CsmDepthDecoderForCausalLM)
Patch 3 applied
Reference: [OpenAI推出Workspace Agents企业可以控制权限和流程把写报告写代码跟进客户等跨系统跨团队的复杂工作流交]
Generate:  [Sesame is a super cool TTS model which can be fine tuned with Unsloth.]
Saved: output_with_context.wav  (3.8s)
